# 00 课程地图：AI+化学实践如何学习

本课程用 8 个短模块回答一个主问题：化学对象如何变成可计算、可建模、可检验的数据任务？

本教程最初为“2026年清华大学 AI for Chemistry 交叉实践暑期课程”准备，现整理为一套自给自足、双语、可公开使用的快速入门材料。

核心路线：`分子/反应 -> 表示 -> 特征 -> 标签 -> 建模 -> 评价 -> 化学理解`

应用场景包括：`分子水溶性预测、结构相似性检索、模型适用域判断、分子空间可视化、反应条件优化`。

这些任务看起来都在“用 AI”，但真正的共同点是：先把化学问题翻译成可建模的数据问题。

## 直觉解释

AI4Chem 不是“把 AI 套到化学上”这么简单。每个任务都要先说清楚：

- 输入是什么：SMILES、descriptor、fingerprint、反应条件。
- 输出是什么：溶解度、产率、活性、毒性。
- 数据是否可信：单位、重复、异常值、实验误差。
- 评价是否公平：random split、scaffold split、时间划分、外部分布。

参考资源：
- RDKit 文档适合查分子表示和 cheminformatics 操作；
- scikit-learn 文档适合查模型、split 和评价指标；
- Delaney ESOL 与 Ahneman/Doyle Buchwald-Hartwig 数据分别对应本课的性质预测和反应优化章节。

## 数学/化学定义

本课程会反复使用这组符号：

```text
X: feature matrix，每一行是一个分子或反应条件
y: label vector，例如 logS 或 yield
f(X): model prediction
error = y_true - y_pred
```

模型不是化学定律。

模型只是从有限数据中学到的近似函数。

In [25]:
from pathlib import Path

START = Path.cwd().resolve()

print("current working directory:", START)

for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

# unpacking the parents of the current working directory into a list and printing them
print("parent directories:", list(START.parents), sep="\n")
print("parent directories:", *START.parents, sep="\n")
# same as above
print("\n".join(map(str, START.parents)))

print("materials root:", ROOT)

current working directory: /home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial/notebooks/zh-CN
parent directories:
[PosixPath('/home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial/notebooks'), PosixPath('/home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial'), PosixPath('/home/isyn/dl/git/whitestone'), PosixPath('/home/isyn/dl/git'), PosixPath('/home/isyn/dl'), PosixPath('/home/isyn'), PosixPath('/home'), PosixPath('/')]
parent directories:
/home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial/notebooks
/home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial
/home/isyn/dl/git/whitestone
/home/isyn/dl/git
/home/isyn/dl
/home/isyn
/home
/
/home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial/notebooks
/home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial
/home/isyn/dl/git/whitestone
/home/isyn/dl/git
/home/isyn/dl
/home/isyn
/home
/
materials root: /home/isyn/dl/git/whitestone/ai4chem-fast-track-tutorial


In [26]:
import pandas as pd

# 课程包含的数据集
examples = pd.read_csv(EXAMPLES / "example_molecules.csv")
esol = pd.read_csv(RAW / "esol.csv")
bh = pd.read_csv(RAW / "buchwald_hartwig.csv")

summary = pd.DataFrame(
    [
        {"file": "example_molecules.csv", "rows": len(examples), "main_use": "SMILES and molecule display"},
        {"file": "esol.csv", "rows": len(esol), "main_use": "solubility prediction"},
        {"file": "buchwald_hartwig.csv", "rows": len(bh), "main_use": "reaction optimization"},
    ]
)
summary

,file,rows,main_use
0,example_molecules.csv,8,SMILES and molecule display
1,esol.csv,1128,solubility prediction
2,buchwald_hartwig.csv,3955,reaction optimization


## 代码

课程包含的模块。

In [3]:
modules = pd.DataFrame(
    [
        ["00", "课程地图", "知道任务框架和学习路线"],
        ["01", "分子如何变成数据", "会读 SMILES、canonical SMILES、分子图"],
        ["02", "描述符与 fingerprint", "理解 descriptor、Morgan fingerprint、Tanimoto"],
        ["03", "ESOL 数据整理", "知道标签、重复、scaffold 和 provenance"],
        ["04", "性质预测", "训练 baseline、Ridge、RandomForestRegressor"],
        ["05", "模型可靠性", "比较 random split 和 scaffold split"],
        ["06", "分子空间", "用 PCA 看高维 fingerprint 的二维投影"],
        ["07", "反应优化", "用 Buchwald-Hartwig 数据理解 surrogate 搜索"],
    ],
    columns=["编号", "主题", "学习目标"],
)
modules

,编号,主题,学习目标
0,00,课程地图,知道任务框架和学习路线
1,01,分子如何变成数据,会读 SMILES、canonical SMILES、分子图
2,02,描述符与 fingerprint,理解 descriptor、Morgan fingerprint、Tanimoto
3,03,ESOL 数据整理,知道标签、重复、scaffold 和 provenance
4,04,性质预测,训练 baseline、Ridge、RandomForestRegressor
5,05,模型可靠性,比较 random split 和 scaffold split
6,06,分子空间,用 PCA 看高维 fingerprint 的二维投影
7,07,反应优化,用 Buchwald-Hartwig 数据理解 surrogate 搜索


## 观察问题

1. 你最熟悉的是哪类输入：分子结构、反应条件、图像、光谱，还是文本？
2. 如果一个模型在测试集 RMSE 很低，它一定能指导真实化学实验吗？
3. 你认为“数据集来源”和“划分方式”哪个更容易被初学者忽略？

### Hints

1. 可以从你最熟悉的化学课内容出发：有机结构式更接近分子图，分析化学数据更接近光谱/图像，实验设计更接近反应条件表。
2. 不一定。要先问测试集是否真的代表未来要预测的分子、反应或实验条件。
3. 两者都容易被忽略。来源决定标签是否可信，划分方式决定模型分数能支持多强的泛化声明。

## 小结

本课程的目标是介绍一个简单但完整的 AI4Chem 工作流：数据从哪里来、分子如何编码、模型如何预测、评价为什么可能误导、结果如何回到化学判断。